# Convergence Check: Clamped E-step with Visible-Only Loss

Scaling factors start perturbed (`weight_perturbation_variance = 0.5`) and we check whether they converge to the target (1.0) over 50 EM iterations.

- **E-step**: Clamped (visible neurons fixed to teacher spikes)
- **Loss**: Visible-only (Van Rossum on visible neurons)
- **Seeds**: 42-46 across two batch runs

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import toml
import zarr

from connectome_snns.utils.reproducibility import load_experiment_config
from connectome_snns.analysis import (
    interleave_spike_trains,
    r_squared,
)
from connectome_snns.analysis.inference import run_recurrent_inference
from connectome_snns.configs.conductance_based import FeedforwardLayerConfig, RecurrentLayerConfig
from connectome_snns.dataloaders.supervised import ExactFFDataset
from connectome_snns.visualization import (
    plot_spike_trains,
    use_project_style,
)
from connectome_snns.visualization.firing_rate_scatter import plot_firing_rate_scatter
from connectome_snns.visualization.scaling_factors import SF_PATHWAYS, plot_sf_trajectories
from connectome_snns.visualization.training_curves import plot_loss_trajectories

use_project_style()

In [ ]:
# Paths and device
convergence_config = load_experiment_config("experiment.toml")
BASE = convergence_config["output_dir"]
SEED_DIRS = {
    42: BASE / "seed-42.0",
    43: BASE / "seed-43.0",
    44: BASE / "seed-44.0",
    45: BASE / "seed-45.0",
    46: BASE / "seed-46.0",
}
PARAMS_FILE = convergence_config["parameters_file"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load parameters
with open(PARAMS_FILE) as f:
    PARAMS = toml.load(f)

RECURRENT_CFG = RecurrentLayerConfig(**PARAMS["recurrent"])
FEEDFORWARD_CFG = FeedforwardLayerConfig(**PARAMS["feedforward"])
RECURRENT_CELL_PARAMS = RECURRENT_CFG.get_cell_params()
FEEDFORWARD_CELL_PARAMS = FEEDFORWARD_CFG.get_cell_params()
RECURRENT_SYNAPSE_PARAMS = RECURRENT_CFG.get_synapse_params()
FEEDFORWARD_SYNAPSE_PARAMS = FEEDFORWARD_CFG.get_synapse_params()
N_FF_CELL_TYPES = len(FEEDFORWARD_CELL_PARAMS)

CHUNK_SIZE = PARAMS["simulation"]["chunk_size"]
SURRGRAD_SCALE = PARAMS["hyperparameters"]["surrgrad_scale"]
HIDDEN_CELL_FRACTION = PARAMS["simulation"]["hidden_cell_fraction"]

# Load network structure (shared across seeds via symlink)
FIRST_SEED_DIR = next(iter(SEED_DIRS.values()))
INPUT_DIR = FIRST_SEED_DIR / "inputs"
NETWORK_STRUCTURE = np.load(INPUT_DIR / "network_structure.npz")
WEIGHTS = NETWORK_STRUCTURE["recurrent_weights"]
FF_WEIGHTS = NETWORK_STRUCTURE["feedforward_weights"]
CELL_TYPE_INDICES = NETWORK_STRUCTURE["cell_type_indices"]
FF_CELL_TYPE_INDICES = NETWORK_STRUCTURE["feedforward_cell_type_indices"]
REC_MASK = NETWORK_STRUCTURE["recurrent_connectivity"]
FF_MASK = NETWORK_STRUCTURE["feedforward_connectivity"]

N_NEURONS_FULL = WEIGHTS.shape[0]
N_HIDDEN = int(N_NEURONS_FULL * HIDDEN_CELL_FRACTION)

# Load teacher spike data
SPIKE_DATASET = ExactFFDataset(
    spike_data_path=INPUT_DIR / "spike_data.zarr",
    chunk_size=CHUNK_SIZE,
    device=DEVICE,
)
DT = SPIKE_DATASET.dt
NUM_CHUNKS = SPIKE_DATASET.num_chunks

TEACHER_ZARR = zarr.open_group(INPUT_DIR / "spike_data.zarr", mode="r")
TEACHER_SPIKES = np.array(TEACHER_ZARR["output_spikes"][:1, :, :])

print(f"Using device: {DEVICE}")
print(
    f"Network: {N_NEURONS_FULL} neurons ({N_HIDDEN} hidden), {FF_WEIGHTS.shape[0]} inputs"
)
print(f"Teacher spikes: {TEACHER_SPIKES.shape}, dt={DT}ms, {NUM_CHUNKS} chunks")

In [ ]:
PARAMS_FILE_PATH = PARAMS_FILE

REC_CELL_TYPE_NAMES = RECURRENT_CFG.cell_types.names
FF_CELL_TYPE_NAMES = FEEDFORWARD_CFG.cell_types.names
COMBINED_INPUT_NAMES = FF_CELL_TYPE_NAMES + REC_CELL_TYPE_NAMES


def _load_learned_sf_matrix(state_dict):
    """Assemble per-pair log_sf scalars into a (n_input_ct, n_output_ct) matrix."""
    n_in = len(COMBINED_INPUT_NAMES)
    n_out = len(REC_CELL_TYPE_NAMES)
    sf = np.ones((n_in, n_out), dtype=np.float64)
    prefix = "ff_projections."
    suffix = ".log_sf"
    for key, val in state_dict.items():
        if not (key.startswith(prefix) and key.endswith(suffix)):
            continue
        pair = key[len(prefix) : -len(suffix)]
        src_name, tgt_name = pair.split("_to_")
        src_idx = COMBINED_INPUT_NAMES.index(src_name)
        tgt_idx = REC_CELL_TYPE_NAMES.index(tgt_name)
        sf[src_idx, tgt_idx] = float(torch.exp(val.detach()).cpu())
    return sf


def run_inference_for_seed(seed, seed_dir, n_analysis_chunks=50):
    """Load learned SFs, run full recurrent model with original weights."""
    target_sf = np.load(seed_dir / "targets" / "target_scaling_factors.npz")[
        "feedforward_scaling_factors"
    ]

    em_dirs = sorted(seed_dir.glob("em_iter_*"))
    ckpt_path = em_dirs[-1] / "checkpoints" / "checkpoint_best.pt"
    if not ckpt_path.exists():
        ckpt_path = em_dirs[-1] / "checkpoints" / "checkpoint_latest.pt"

    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    learned_sf = _load_learned_sf_matrix(ckpt["model_state_dict"])

    effective_sf = learned_sf / target_sf
    print(f"  Effective SF (should be ~1.0): {effective_sf.flatten()}")

    effective_sf_ff = effective_sf[:N_FF_CELL_TYPES]
    effective_sf_rec = effective_sf[N_FF_CELL_TYPES:]

    result = run_recurrent_inference(
        params_file=PARAMS_FILE_PATH,
        run_dir=seed_dir,
        scaling_factors=effective_sf_rec,
        scaling_factors_FF=effective_sf_ff,
        n_burnin_chunks=0,
        n_analysis_chunks=n_analysis_chunks,
        device=DEVICE,
        desc=f"seed {seed}",
    )
    return result["student_spikes"][None, ...]

In [ ]:
# Load training metrics from all complete seeds
metrics = {}
for seed, path in SEED_DIRS.items():
    csv_path = path / "training_metrics.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        if len(df) > 10:  # skip incomplete runs
            metrics[seed] = df
            print(
                f"Seed {seed}: {len(df)} rows, epochs {df['epoch'].iloc[0]}-{df['epoch'].iloc[-1]}"
            )
    else:
        print(f"Seed {seed}: MISSING")

print(f"\n{len(metrics)} complete seeds loaded")

## Scaling Factor Convergence

In [ ]:
fig = plot_sf_trajectories(
    metrics,
    pathways=SF_PATHWAYS,
    label_fn=lambda s: f"Seed {s}",
    suptitle="Scaling Factor Convergence (Perturbed \u2192 Target)",
    figsize=(12, 10),
)
plt.show()

In [ ]:
# Summary: initial and final scaling factor values
rows = []
for key, title in SF_PATHWAYS:
    col_val = f"scaling_factors/{key}_value"
    initials = [df[col_val].iloc[0] for df in metrics.values() if col_val in df.columns]
    finals = [df[col_val].iloc[-1] for df in metrics.values() if col_val in df.columns]
    rows.append(
        {
            "Pathway": title,
            "Initial (mean \u00b1 std)": f"{np.mean(initials):.3f} \u00b1 {np.std(initials):.3f}",
            "Final (mean \u00b1 std)": f"{np.mean(finals):.3f} \u00b1 {np.std(finals):.3f}",
            "Target": 1.0,
            "Final Error (%)": f"{np.mean(np.abs(np.array(finals) - 1.0)) * 100:.1f}",
        }
    )

pd.DataFrame(rows).style.set_caption("Scaling Factor Summary")

## Loss Convergence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_loss_trajectories(
    metrics,
    "total_loss",
    title="Total Loss",
    label_fn=lambda s: f"Seed {s}",
    ax=axes[0],
)
plot_loss_trajectories(
    metrics,
    "van_rossum_loss",
    title="Van Rossum Loss",
    ylabel="Van Rossum Loss",
    label_fn=lambda s: f"Seed {s}",
    ax=axes[1],
)

plt.suptitle("Loss Convergence", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Firing Rate Evolution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

# (row, col): (visibility, cell_type)
panels = [
    (0, 0, "visible", "excitatory", "Visible Excitatory"),
    (0, 1, "visible", "inhibitory", "Visible Inhibitory"),
    (1, 0, "hidden", "excitatory", "Hidden Excitatory"),
    (1, 1, "hidden", "inhibitory", "Hidden Inhibitory"),
]

for row, col, vis, cell_type, title in panels:
    ax = axes[row, col]
    student_col = f"firing_rate/student_{vis}_{cell_type}_mean"
    teacher_col = f"firing_rate/teacher_{vis}_{cell_type}_mean"

    for seed, df in metrics.items():
        if student_col in df.columns:
            ax.plot(
                df["epoch"],
                df[student_col],
                linewidth=1,
                alpha=0.7,
                label=f"Seed {seed}",
            )

    # Teacher reference
    first_df = next(iter(metrics.values()))
    if teacher_col in first_df.columns:
        ax.plot(
            first_df["epoch"],
            first_df[teacher_col],
            color="black",
            linestyle="--",
            linewidth=1.5,
            label="Teacher",
        )

    ax.set_title(title, fontsize=11)
    ax.set_ylabel("Firing Rate (Hz)")
    ax.set_ylim(0, None)
    if row == 1:
        ax.set_xlabel("Epoch")
    if row == 0 and col == 0:
        ax.legend(loc="best", fontsize=7)

plt.suptitle("Firing Rate Evolution", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Gradient Norms

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# TODO: confirm wandb column naming. Old single column
# `gradients/log_scaling_factors_FF_norm` is replaced by per-pair columns
# `gradients/ff_projections.<src>_to_<tgt>.log_sf_norm` (one per projection).
# Here we aggregate into a single norm by sqrt(sum(per_pair_norm**2)).
grad_prefix = "gradients/ff_projections."
grad_suffix = ".log_sf_norm"
for seed, df in metrics.items():
    pair_cols = [
        c for c in df.columns if c.startswith(grad_prefix) and c.endswith(grad_suffix)
    ]
    if not pair_cols:
        continue
    agg = np.sqrt((df[pair_cols] ** 2).sum(axis=1))
    ax.plot(df["epoch"], agg, linewidth=1, alpha=0.7, label=f"Seed {seed}")

ax.set_xlabel("Epoch")
ax.set_ylabel("Gradient Norm")
ax.set_title(
    "Log Scaling Factor Gradient Norm (FF, aggregated)", fontsize=14, fontweight="bold"
)
ax.set_ylim(0, None)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Summary

In [ ]:
# Per-seed summary: final loss, final SF error, final firing rates
rows = []
for seed, df in metrics.items():
    last = df.iloc[-1]

    # Mean absolute SF error across all 6 pathways
    sf_errors = []
    for key, _ in SF_PATHWAYS:
        col_val = f"scaling_factors/{key}_value"
        if col_val in df.columns:
            sf_errors.append(abs(last[col_val] - 1.0))

    rows.append(
        {
            "Seed": seed,
            "Final Loss": f"{last['total_loss']:.2f}",
            "Final VR Loss": f"{last['van_rossum_loss']:.2f}",
            "Mean SF Error": f"{np.mean(sf_errors):.4f}",
            "Max SF Error": f"{np.max(sf_errors):.4f}",
            "Vis Exc FR (Hz)": f"{last.get('firing_rate/student_visible_excitatory_mean', np.nan):.1f}",
            "Vis Inh FR (Hz)": f"{last.get('firing_rate/student_visible_inhibitory_mean', np.nan):.1f}",
        }
    )

pd.DataFrame(rows).style.set_caption("Per-Seed Final Metrics")

## Model Inference with Learned Scaling Factors

Run the full recurrent model with the final learned scaling factors for each seed, and compare output spike trains and firing rates to the teacher.

In [ ]:
# Run inference for seed 42
seed_to_run = 42
print(f"Running inference for seed {seed_to_run}...")
student_spikes = run_inference_for_seed(seed_to_run, SEED_DIRS[seed_to_run])
print(f"Student spikes: {student_spikes.shape}")

In [ ]:
# Spike raster: teacher vs student
min_time = min(student_spikes.shape[1], TEACHER_SPIKES.shape[1])
n_neurons_plot = 15

exc_neurons = np.where(CELL_TYPE_INDICES == 0)[0][:n_neurons_plot]
plot_idx = exc_neurons

teacher_subset = TEACHER_SPIKES[0][:min_time, plot_idx]
student_subset = student_spikes[0][:min_time, plot_idx]

interleaved, ct_idx = interleave_spike_trains(teacher_subset, student_subset)

fig = plot_spike_trains(
    spikes=interleaved,
    dt=DT,
    cell_type_indices=ct_idx,
    cell_type_names=["Teacher", "Student"],
    n_neurons_plot=2 * len(plot_idx),
    n_compared=2,
    fraction=1.0,
    random_seed=None,
    title=f"Teacher vs Student (Seed {seed_to_run})",
    ylabel="Neuron",
    figsize=(16, 8),
)

In [ ]:
# Per-neuron firing rate comparison
min_time = min(student_spikes.shape[1], TEACHER_SPIKES.shape[1])
duration_s = min_time * DT / 1000.0

teacher_rates = TEACHER_SPIKES[0][:min_time, :].sum(axis=0) / duration_s
student_rates = student_spikes[0][:min_time, :].sum(axis=0) / duration_s

exc_mask = CELL_TYPE_INDICES == 0
inh_mask = CELL_TYPE_INDICES == 1

r2 = r_squared(teacher_rates, student_rates)

fig = plot_firing_rate_scatter(
    teacher_rates,
    student_rates,
    CELL_TYPE_INDICES,
    r2=r2,
    title=f"Per-Neuron Firing Rates (R\u00b2 = {r2:.3f}, Seed {seed_to_run})",
)
plt.tight_layout()
plt.show()

# Print summary
print(
    f"Teacher: exc={teacher_rates[exc_mask].mean():.1f}\u00b1{teacher_rates[exc_mask].std():.1f} Hz, "
    f"inh={teacher_rates[inh_mask].mean():.1f}\u00b1{teacher_rates[inh_mask].std():.1f} Hz"
)
print(
    f"Student: exc={student_rates[exc_mask].mean():.1f}\u00b1{student_rates[exc_mask].std():.1f} Hz, "
    f"inh={student_rates[inh_mask].mean():.1f}\u00b1{student_rates[inh_mask].std():.1f} Hz"
)